In [1]:
import numpy as np
import random

# =========================================================
# QUIZ 2 - MODELE CIR
# =========================================================
# Ordre des questions :
# Q1  : Simuler 1000 trajectoires CIR (Monte-Carlo)
# Q2  : Statistiques descriptives de r(365)
# Q3a : Prime du caplet forward-looking (Monte-Carlo)
# Q3b : Prime du caplet backward-looking (Monte-Carlo)
# Q4  : Construire la grille explicite en différences finies
# Q5  : Évaluer les puts européen et américain sur ZC
# =========================================================

In [22]:
# =========================================================
# 0) PARAMETRES A REMPLACER LE JOUR DU QUIZ
# =========================================================

# Paramètres CIR
kappa = 0.70
theta = 0.05
sigma = 0.12
r0 = 0.03

# Seed donnée au quiz
seed_value = 2026

# ---------- Q1 à Q3 : Monte-Carlo ----------
n_paths = 1000
n_days = 365
dt_sim = 1 / 365

fixing_day = 275
payment_day = 365
K_caplet = 0.045
tau_caplet = (payment_day - fixing_day) / 365  # 90/365

# ---------- Q4 à Q5 : Différences finies ----------
dr = 0.001
Nr = 100                  # 100 pas d'espace => 101 noeuds
dt_fd = 1 / 2000
Nt = 2000                 # 2000 pas de temps

T_put = 1.0               # échéance du put
T_bond = 3.0              # maturité de la ZC
K_put = 0.96              # strike du put


In [23]:
# =========================================================
# 1) FONCTIONS UTILES
# =========================================================

def cir_zcb_price(r, tau, kappa, theta, sigma):
    """
    Prix de l'obligation zéro-coupon CIR :
    P(t,T) = a(t,T) * exp(-b(t,T) * r_t)
    """
    gamma = np.sqrt(kappa**2 + 2 * sigma**2)

    a_num = 2 * gamma * np.exp((kappa + gamma) * tau / 2)
    a_den = (kappa + gamma) * (np.exp(gamma * tau) - 1) + 2 * gamma
    a = (a_num / a_den) ** (2 * kappa * theta / sigma**2)

    b = 2 * (np.exp(gamma * tau) - 1) / ((kappa + gamma) * (np.exp(gamma * tau) - 1) + 2 * gamma)

    return a * np.exp(-b * r)


def simulate_cir_euler(kappa, theta, sigma, r0, n_paths, n_steps, dt, seed_value):
    """
    Simulation Euler du processus CIR.
    Consigne du quiz : pas besoin de corriger les valeurs négatives.
    """
    random.seed(seed_value)
    np.random.seed(seed_value)

    rates = np.zeros((n_paths, n_steps + 1))
    rates[:, 0] = r0

    for t in range(n_steps):
        z = np.random.normal(size=n_paths)

        # on garde la logique du quiz : Euler simple
        # on protège seulement le sqrt pour éviter une erreur numérique
        sqrt_term = np.sqrt(np.maximum(rates[:, t], 0.0))

        rates[:, t + 1] = (
            rates[:, t]
            + kappa * (theta - rates[:, t]) * dt
            + sigma * sqrt_term * np.sqrt(dt) * z
        )

    return rates


def descriptive_stats(x):
    return {
        "moyenne": np.mean(x),
        "ecart_type": np.std(x, ddof=1),
        "Q1": np.quantile(x, 0.25),
        "Q2_mediane": np.quantile(x, 0.50),
        "Q3": np.quantile(x, 0.75)
    }

## Question 1

In [24]:
# =========================================================
# Q1) SIMULATION DE 1000 TRAJECTOIRES CIR
# =========================================================

rates = simulate_cir_euler(
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    r0=r0,
    n_paths=n_paths,
    n_steps=n_days,
    dt=dt_sim,
    seed_value=seed_value
)

print("=== Q1 : Simulation CIR complétée ===")
print("Dimension de la matrice :", rates.shape)


=== Q1 : Simulation CIR complétée ===
Dimension de la matrice : (1000, 366)


## Question 2

In [26]:
# =========================================================
# Q2) STATISTIQUES DESCRIPTIVES DE r(365)
# =========================================================

# on extrait la distribution du taux après 365 jours
r_365 = rates[:, 365]

print("Moyenne de r(365) :", np.mean(r_365))

stats_365 = descriptive_stats(r_365)

print("\n=== Q2 : Statistiques descriptives de r(365) ===")
for key, value in stats_365.items():
    print(f"{key}: {value:.6f}")

Moyenne de r(365) : 0.040201774860618794

=== Q2 : Statistiques descriptives de r(365) ===
moyenne: 0.040202
ecart_type: 0.016828
Q1: 0.027422
Q2_mediane: 0.037836
Q3: 0.051391


## Question 3

In [15]:
# =========================================================
# Q3a) CAPLET FORWARD-LOOKING
# =========================================================
#
# Payoff caplet :
# tau * max(i_{T1}(T2) - K, 0)
#
# On approxime le taux forward-looking par le taux simple implicite
# observé à T1 sur [T1,T2] :
#
# i_T1(T2) = (1 / P(T1,T2) - 1) / tau
#
# Puis on actualise trajectoire par trajectoire.
# =========================================================

caplet_fw_discounted = np.zeros(n_paths)

for i in range(n_paths):
    r_T1 = rates[i, fixing_day]

    P_T1_T2 = cir_zcb_price(r_T1, tau_caplet, kappa, theta, sigma)
    i_forward = (1 / P_T1_T2 - 1) / tau_caplet

    payoff_T2 = tau_caplet * max(i_forward - K_caplet, 0)

    discount_0_T2 = np.exp(-np.sum(rates[i, :payment_day]) * dt_sim)

    caplet_fw_discounted[i] = discount_0_T2 * payoff_T2

caplet_fw_price = np.mean(caplet_fw_discounted)
caplet_fw_bps = 10000 * caplet_fw_price

print("\n=== Q3a : Caplet forward-looking ===")
print(f"Prix: {caplet_fw_price:.8f}")
print(f"Prix en points de base: {caplet_fw_bps:.4f}")



=== Q3a : Caplet forward-looking ===
Prix: 0.00063078
Prix en points de base: 6.3078


In [16]:
# =========================================================
# Q3b) CAPLET BACKWARD-LOOKING
# =========================================================
#
# D'après les notes de cours, pour un taux backward-looking
# sur overnight aggregated rate, le payoff est basé sur
# la moyenne arithmétique du taux court simulé sur la période.
#
# r_bar = moyenne des taux simulés entre jours 275 et 365
# payoff = tau * max(r_bar - K, 0)
# =========================================================

caplet_bw_discounted = np.zeros(n_paths)

for i in range(n_paths):
    period_rates = rates[i, fixing_day:payment_day]
    r_bar = np.mean(period_rates)

    payoff_T2 = tau_caplet * max(r_bar - K_caplet, 0)

    discount_0_T2 = np.exp(-np.sum(rates[i, :payment_day]) * dt_sim)

    caplet_bw_discounted[i] = discount_0_T2 * payoff_T2

caplet_bw_price = np.mean(caplet_bw_discounted)
caplet_bw_bps = 10000 * caplet_bw_price

print("\n=== Q3b : Caplet backward-looking ===")
print(f"Prix: {caplet_bw_price:.8f}")
print(f"Prix en points de base: {caplet_bw_bps:.4f}")


=== Q3b : Caplet backward-looking ===
Prix: 0.00075849
Prix en points de base: 7.5849


## Question 4

In [17]:
# =========================================================
# Q4) CONSTRUCTION DE LA GRILLE EFD
# =========================================================

r_grid = np.arange(Nr + 1) * dr
t_grid = np.arange(Nt + 1) * dt_fd

print("\n=== Q4 : Grille EFD construite ===")
print(f"Nombre de noeuds en r : {len(r_grid)}")
print(f"Nombre de noeuds en t : {len(t_grid)}")
print(f"r_max = {r_grid[-1]:.6f}")


=== Q4 : Grille EFD construite ===
Nombre de noeuds en r : 101
Nombre de noeuds en t : 2001
r_max = 0.100000


## Question 5

In [18]:
# =========================================================
# Q5) PUTS EUROPEEN ET AMERICAIN SUR ZC
# =========================================================
#
# Sous-jacent : obligation zéro-coupon d'échéance 3 ans
# Option : put de maturité 1 an
#
# À t = T_put, la valeur du sous-jacent est P(T_put, T_bond)
# avec maturité résiduelle T_bond - T_put = 2 ans.
#
# Conditions aux bornes inspirées directement des notes :
#
# A maturité :
# g_i^T = max(0, K - P_cir(T_put, T_bond, r_i))
#
# A r = 0 :
# g_0^t = max(0, K P_cir(0, T_put, 0) - P_cir(0, T_bond, 0))
#
# A r = r_max :
# g_imax^t = max(0, K P_cir(0, T_put, r_max) - P_cir(0, T_bond, r_max))
#
# Pour l'américain :
# max(valeur de continuation, valeur d'exercice)
# =========================================================

V_eur = np.zeros((Nt + 1, Nr + 1))
V_ame = np.zeros((Nt + 1, Nr + 1))

# Condition terminale à t = T_put
bond_at_maturity = cir_zcb_price(r_grid, T_bond - T_put, kappa, theta, sigma)
payoff_put = np.maximum(K_put - bond_at_maturity, 0.0)

V_eur[Nt, :] = payoff_put
V_ame[Nt, :] = payoff_put

# Boucle backward
for n in range(Nt - 1, -1, -1):
    t = n * dt_fd

    # temps restant vers la maturité du bond à la date t
    tau_bond_now = T_bond - t
    bond_now = cir_zcb_price(r_grid, tau_bond_now, kappa, theta, sigma)

    # exercice immédiat
    intrinsic_now = np.maximum(K_put - bond_now, 0.0)

    # Conditions aux bornes selon la logique des notes
    r_max = r_grid[-1]

    left_boundary = max(0.0,
                        K_put * cir_zcb_price(0.0, T_put - t, kappa, theta, sigma)
                        - cir_zcb_price(0.0, T_bond - t, kappa, theta, sigma))

    right_boundary = max(0.0,
                         K_put * cir_zcb_price(r_max, T_put - t, kappa, theta, sigma)
                         - cir_zcb_price(r_max, T_bond - t, kappa, theta, sigma))

    V_eur[n, 0] = left_boundary
    V_ame[n, 0] = left_boundary

    V_eur[n, Nr] = right_boundary
    V_ame[n, Nr] = max(right_boundary, intrinsic_now[Nr])

    for j in range(1, Nr):
        r = r_grid[j]

        mu = kappa * (theta - r)
        var = sigma**2 * r

        A = 0.5 * dt_fd * (var / dr**2 - mu / dr)
        B = 1 - dt_fd * (var / dr**2 + r)
        C = 0.5 * dt_fd * (var / dr**2 + mu / dr)

        continuation_eur = (
            A * V_eur[n + 1, j - 1]
            + B * V_eur[n + 1, j]
            + C * V_eur[n + 1, j + 1]
        )

        continuation_ame = (
            A * V_ame[n + 1, j - 1]
            + B * V_ame[n + 1, j]
            + C * V_ame[n + 1, j + 1]
        )

        V_eur[n, j] = continuation_eur
        V_ame[n, j] = max(continuation_ame, intrinsic_now[j])

# Prix au temps 0
eur_put_price = np.interp(r0, r_grid, V_eur[0, :])
ame_put_price = np.interp(r0, r_grid, V_ame[0, :])

eur_put_bps = 10000 * eur_put_price
ame_put_bps = 10000 * ame_put_price

print("\n=== Q5 : Put européen sur zéro-coupon ===")
print(f"Prix: {eur_put_price:.8f}")
print(f"Prix en points de base: {eur_put_bps:.4f}")

print("\n=== Q5 : Put américain sur zéro-coupon ===")
print(f"Prix: {ame_put_price:.8f}")
print(f"Prix en points de base: {ame_put_bps:.4f}")


=== Q5 : Put européen sur zéro-coupon ===
Prix: 0.01918777
Prix en points de base: 191.8777

=== Q5 : Put américain sur zéro-coupon ===
Prix: 0.04868960
Prix en points de base: 486.8960
